# Qwen2.5-1.5B task-extraction SFT — Hybrid Architecture (v8)

Fine-tunes the model Xayra ships for on-device to-do extraction.

## Why this notebook is standard SFT again, not ORPO

A 103-case evaluation harness put the prompt-only baseline at **83.5%**, with two failures prompt work alone could not move: third-party attribution (0/8) and zero-task refusal (11/17, i.e. the model invented tasks from purely descriptive notes).

Five real training attempts tried to fix this in the model itself:

| Run | Approach | Result |
|---|---|---|
| 2 | Plain SFT, 50% refusal | Fixed third-party/refusal, broke dates/recurrence |
| 3 | Plain SFT, 15% refusal | Fixed dates/recurrence, broke third-party/refusal again |
| 5 | ORPO, third-party in a shared refusal bucket | Third-party collapsed to 1/8 — a NEW failure (paraphrasing the visitor's action as a task) |
| 6 | ORPO, third-party bumped 26→36, mixed rejected shapes | Trained and re-swept: **still 1/8**, nearly identical output to v5 |
| 7 | ORPO, third-party 36→70, LoRA r=32, 3 epochs, warmup fix | Trained and run against the FULL 103-case harness: **68.9%, below the 83.5% baseline** — third-party still 1/8, refusal collapsed to 5/17 |

v7's actual failure output was the deciding evidence: its spurious tasks were near-verbatim reproductions of this file's OWN hand-authored `rejected` fabrication strings ("Have the plumber come", "Move house", "Collect green waste"). Showing the model a small, fixed vocabulary of "wrong" answers — even as ORPO's disfavoured target — taught it those phrases are generically plausible outputs, not that third-party notes have no task at all.

**The Hybrid Architecture accepts this as a real ceiling on what fine-tuning should do here.** Third-party/observation refusal is now a DETERMINISTIC pre-filter in code (`preFilterZeroTaskNotes` in `services/ai/extractionLogic.ts`) — verified against the frozen 103-case corpus with zero false positives, catching 25 of 26 known zero-task cases, before this notebook was touched. Fine-tuning is scoped down to exactly what every prior run got right every time: dates, recurrence, STT-noise tolerance, contrastive skip-the-distractor extraction, and RAG grounding/refusal. With no refusal examples left to train, there's nothing for ORPO's preference mechanism to score against — this notebook reverts to a standard `SFTTrainer` + `train_on_responses_only` (masks the loss to assistant-turn tokens only). Not yet trained or evaluated.

## Runtime

Free Colab **T4**. Set `Runtime -> Change runtime type -> T4 GPU` before running. Roughly 15-20 minutes end to end for 500 rows over 2 epochs.

## 1. Dependencies

Unsloth pins compatible `peft`/`xformers` builds itself; installing them loose alongside it is the usual cause of a broken Colab session. `trl`'s standard `SFTTrainer`/`SFTConfig` (not the `trl.experimental` namespace ORPO lived in) are a stable, long-established API — the version churn that hit `ORPOConfig` twice in this project's history does not apply here.

In [ ]:
%%capture
import torch

major, _ = torch.cuda.get_device_capability()

!pip install -q --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

if major >= 8:
    # Ampere and newer (A100, L4): flash-attn and bf16 are available.
    !pip install -q --no-deps packaging ninja einops flash-attn xformers trl peft accelerate bitsandbytes
else:
    # T4 is Turing (sm75): no flash-attn, no bf16. Unsloth falls back to fp16.
    !pip install -q --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
import torch

print("GPU        :", torch.cuda.get_device_name(0))
print("Capability :", torch.cuda.get_device_capability())
print("VRAM (GB)  :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print("bf16       :", torch.cuda.is_bf16_supported())

## 2. Dataset

Upload `sft_qwen_task_extraction.jsonl`, produced by `scripts/dataset/generate_sft.py` (v8). Each row is `{"messages": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}], "kind": ...}` — plain chat messages, no `prompt`/`chosen`/`rejected` split, since there is no rejected side any more (see the module docstring's "WHY V8" section for why ORPO's negative-example approach was abandoned for this dataset).

500 rows total: 350 extraction (270 positive, 80 contrastive, **0 refusal**) + 150 RAG (75 factual, 75 refusal). Third-party/observation refusal training is gone entirely — that behaviour is now a deterministic code-side pre-filter (`preFilterZeroTaskNotes`), not something this fine-tune attempts. The freed budget went to `extraction_positive_other` (184 → 270), with date coverage raised to ≥70% and recurrence to ≥25%.

In [ ]:
from google.colab import files

uploaded = files.upload()  # select sft_qwen_task_extraction.jsonl
DATASET_PATH = next(iter(uploaded))
print("Using:", DATASET_PATH)

In [ ]:
import json
from collections import Counter
from datasets import Dataset

rows = [json.loads(line) for line in open(DATASET_PATH, encoding="utf-8") if line.strip()]

# Every row must carry a real 3-turn conversation -- a file that silently
# lost a turn (the same class of schema drift that broke this notebook's
# ORPO-era validation cell before) should fail HERE, not after a training
# run.
for i, r in enumerate(rows):
    assert "messages" in r, f"row {i} missing 'messages'"
    roles = [m["role"] for m in r["messages"]]
    assert roles == ["system", "user", "assistant"], f"row {i} has roles {roles}, expected system/user/assistant"
    assert all(m["content"].strip() for m in r["messages"]), f"row {i} has an empty message"

counts = Counter(r["kind"] for r in rows)
total = len(rows)
print(f"rows             : {total}")
for kind in sorted(counts):
    print(f"  {kind:24s} {counts[kind]:4d}  ({counts[kind] / total:.0%})")

# v8: every extraction row must have a non-empty assistant JSON array --
# there are NO refusal ("[]") examples in this dataset at all (that
# behaviour is now a code-side pre-filter, not something this fine-tune
# attempts). A stray "[]" here would mean the dataset regenerated with the
# old v5-v7 refusal buckets by mistake.
extraction_rows = [r for r in rows if r["kind"] in ("extraction_positive", "extraction_contrastive")]
empty_extraction = [r for r in extraction_rows if r["messages"][-1]["content"].strip() == "[]"]
print()
print(f"extraction rows with an empty '[]' answer : {len(empty_extraction)}  (must be 0)")
assert len(empty_extraction) == 0, "found a refusal-shaped ('[]') extraction row -- v8 should contain none"

required_kinds = {"extraction_positive", "extraction_contrastive", "rag_factual", "rag_refusal"}
missing_kinds = required_kinds - set(counts)
assert not missing_kinds, f"missing kinds: {missing_kinds}"
assert counts["extraction_contrastive"] >= 75, f"only {counts['extraction_contrastive']} contrastive rows, need >=75"
assert counts["extraction_positive"] >= 200, f"only {counts['extraction_positive']} positive rows, need >=200"
# v8.1: RAG rebalanced 75/75 -> 125/25 (85/15) -- the v8 checkpoint
# over-refused on genuinely-answerable INDIRECT questions, and cutting
# refusal volume is part of the fix. Floors updated to match; the old
# ">=70" rag_refusal floor would fail on this dataset by design now.
assert counts["rag_factual"] >= 110, f"only {counts['rag_factual']} rag_factual rows, need >=110"
assert counts["rag_refusal"] >= 15, f"only {counts['rag_refusal']} rag_refusal rows, need >=15"

dataset = Dataset.from_list([{"messages": r["messages"]} for r in rows])
print()
print("--- sample row ---")
for m in dataset[0]["messages"]:
    print(f"[{m['role']}] {m['content']}")

## 3. Model — QLoRA via Unsloth

`FastLanguageModel.from_pretrained`/`get_peft_model` load and adapt the base model the same way regardless of which trainer updates the LoRA weights afterward. `r=16` with `lora_alpha=16` (1:1, reverted from v7's `r=32`) — v7's extra capacity existed specifically to help a minority-pattern rule (third-party refusal) survive against a dominant "extract a task" signal; v8 trains no such minority-pattern behaviour at all (see the dataset section above), so the standard v5 configuration is the right baseline again, not a compromise. `max_seq_length=1024` (down from 2048) matches the directive's training config — this dataset's rows (single-turn extraction/RAG conversations, no dual-completion ORPO budget to split) comfortably fit.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,      # None lets Unsloth pick fp16 on T4, bf16 on Ampere+
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,          # 0 is Unsloth's optimised path
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=20260921,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params: {trainable:,}")

## 4. Train — standard SFT

`learning_rate=8e-5`, `num_train_epochs=2`, `warmup_ratio=0.1`, `max_seq_length=1024` — the directive's specified hyperparameters. `SFTConfig`/`SFTTrainer` are trl's standard, non-experimental classes (unlike ORPO's `trl.experimental.orpo`), so `warmup_ratio` is a normal, always-accepted `TrainingArguments` field here — no defensive introspection needed the way `ORPOConfig` required.

Three steps the dataset needs before `SFTTrainer` can use it, confirmed against Unsloth's own docs and a real Qwen2.5 reference notebook before writing this cell (same discipline `PatchDPOTrainer` got in v4):

1. `get_chat_template(tokenizer, chat_template="qwen-2.5")` — sets the tokenizer's chat template to Qwen2.5's real ChatML markers, needed both for formatting below and for `train_on_responses_only` to find the right split points.
2. `dataset.map(...)` applies `tokenizer.apply_chat_template` to each row's `messages` list, producing a single `text` string column — `SFTTrainer` does not template `messages` for you.
3. `train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")`, called AFTER the trainer is constructed and BEFORE `.train()` — masks the loss to assistant-turn tokens only, so the model is never trained to predict its own system/user turns. These exact strings are Qwen2.5 ChatML's real markers, not Llama-3's `<|start_header_id|>` format.

In [ ]:
from unsloth.chat_templates import get_chat_template, train_on_responses_only

# Sets tokenizer.chat_template to Qwen2.5's real ChatML template -- must run
# before both the dataset formatting below and train_on_responses_only,
# which finds its split points from this template.
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in examples["messages"]
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print("--- formatted sample ---")
print(dataset[0]["text"])

from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,   # effective batch size 8
        num_train_epochs=2,
        learning_rate=8e-5,
        warmup_ratio=0.1,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        seed=20260921,
        output_dir="outputs",
        report_to="none",
    ),
)

# Masks the loss to assistant-turn tokens only -- called after the trainer
# is built, before .train(). Reassign the return value; it patches the
# trainer's data collator rather than mutating in place.
sft_trainer = train_on_responses_only(
    sft_trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

stats = sft_trainer.train()
print(stats)

## 5. Smoke test before export

Four prompts covering behaviours this fine-tune actually trains: a dated task, a recurring task, a contrastive note, and a multi-task note. Third-party/observation refusal is deliberately NOT tested here — that's now `preFilterZeroTaskNotes`'s job in code, not this model's, and testing it against the raw model would be checking a behaviour this dataset no longer teaches at all (see section 2 above). Worth thirty seconds here rather than discovering a collapsed model after a 1GB download — but a clean pass here is NOT proof the fix worked: this project's history has a clean smoke test pass then fail the full harness more than once. Section 6 below is the real pre-export gate.

Expected: a single task, a recurring task, only the real task (not the distractor), two tasks.

In [ ]:
SYSTEM_PROMPT = (
    "You are an executive task extraction assistant. Extract actionable user "
    "tasks into the requested JSON schema. If no tasks exist for the user, "
    "return []."
)

RAG_SYSTEM_PROMPT = (
    "You are Xayra, an on-device notes assistant. Answer the question using "
    "ONLY the note context provided below. If the answer is not in the "
    "notes, reply exactly: \"No information found in your notes.\""
)

PROBES = [
    ("Renew the parking permit by Friday.", "one dated task"),
    ("Water the office plants every Friday.", "one recurring task"),
    ("The electrician is coming Tuesday to check the wiring. Buy lightbulbs tomorrow.", "only the lightbulbs task"),
    ("Book the flights and renew the travel insurance.", "two tasks"),
]

FastLanguageModel.for_inference(model)

# Uses the SAME tokenizer.apply_chat_template path training used (via
# get_chat_template in section 4), rather than a hand-rolled ChatML string
# -- removes any risk of a subtle formatting mismatch between what was
# trained on and what the smoke test/sweep actually sends.
def generate(user_content, system_prompt=SYSTEM_PROMPT):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=128, temperature=0.0, do_sample=False)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

def generate_rag(user_content):
    return generate(user_content, system_prompt=RAG_SYSTEM_PROMPT)

for note, expectation in PROBES:
    answer = generate(note)
    print(f"{note}\n  expect: {expectation}\n  got   : {answer}\n")

## 6. Automated 20-case validation sweep

Scoped to exactly what v8 trains — 6 dated tasks, 4 recurring tasks, 6 contrastive (task + distractor together), 4 RAG (2 factual, 2 refusal). Third-party/observation refusal is intentionally absent — see section 5's note on why. None of these sentences exist in the training file.

**This is a gate, not the verdict.** A clean sweep here does not replace running the exported GGUF through the real 103-case harness afterward — that harness also exercises `preFilterZeroTaskNotes` for third-party/refusal, which this notebook cannot test at all (it's TypeScript, not something the raw model does). Treat a pass here as "worth spending the export time," not as "done."

In [ ]:
import json as _json

# (user_content, category, check) -- `check` takes the raw generated text and
# returns bool. Kept intentionally simpler than the real TypeScript harness
# (scripts/eval/scoring.ts) -- this only has to decide "is this checkpoint
# worth exporting," not produce a publishable score.

def _has_a_task(text):
    try:
        parsed = _json.loads(text)
        return isinstance(parsed, list) and len(parsed) > 0
    except Exception:
        return False

def _has_task_without_leaking(forbidden):
    def _check(text):
        if not _has_a_task(text):
            return False
        lower = text.lower()
        return not any(word.lower() in lower for word in forbidden)
    return _check

def _contains_facts(facts):
    def _check(text):
        lower = text.lower()
        return all(fact.lower() in lower for fact in facts)
    return _check

def _is_rag_refusal(text):
    return "no information found" in text.lower()

VALIDATION_CASES = [
    # -- dated tasks (6): must extract a task, must not be empty --
    ("Renew the parking permit by Friday.", "date", "extraction", _has_a_task),
    ("Call the accountant next Tuesday about the invoice.", "date", "extraction", _has_a_task),
    ("Pay the storage fee on the 5th of every month.", "date", "extraction", _has_a_task),
    ("Collect the dry cleaning this Saturday.", "date", "extraction", _has_a_task),
    ("Book the vet appointment before the end of the month.", "date", "extraction", _has_a_task),
    ("Chase the delivery slot on the 12th.", "date", "extraction", _has_a_task),
    # -- recurring tasks (4): must extract a task with recurrence set --
    ("Take the vitamins every morning.", "recurrence", "extraction", _has_a_task),
    ("Submit the timesheet every Friday.", "recurrence", "extraction", _has_a_task),
    ("Back up the photos every month.", "recurrence", "extraction", _has_a_task),
    ("Check in with the team every other week.", "recurrence", "extraction", _has_a_task),
    # -- contrastive (6): must extract ONLY the real task, distractor must not leak --
    ("The electrician is coming Tuesday to check the wiring. Buy lightbulbs tomorrow.",
     "contrastive", "extraction", _has_task_without_leaking(["electrician", "wiring"])),
    ("My cousin is moving to Berlin next month. Cancel the newspaper subscription.",
     "contrastive", "extraction", _has_task_without_leaking(["cousin", "Berlin"])),
    ("The new cafe down the street is fantastic. Water the plants every Tuesday.",
     "contrastive", "extraction", _has_task_without_leaking(["cafe"])),
    ("The neighbours are getting their roof redone this week. Book the dentist for next Wednesday.",
     "contrastive", "extraction", _has_task_without_leaking(["neighbours", "roof"])),
    ("Traffic was much lighter than usual this morning. Renew the passport before the trip.",
     "contrastive", "extraction", _has_task_without_leaking(["traffic"])),
    ("The plumber said he'd call round on Friday. Chase the refund from the airline.",
     "contrastive", "extraction", _has_task_without_leaking(["plumber"])),
    # -- RAG factual (2): must state the grounded fact --
    ("--- NOTE 1 [Recorded: Monday, 12 Jan 2026 at 09:00] ---\nYour dentist appointment is booked for 15 January at 2pm.\n\nQuestion: when is my dentist appointment?",
     "rag_factual", "rag", _contains_facts(["15 January", "2pm"])),
    ("--- NOTE 1 [Recorded: Tuesday, 03 Feb 2026 at 10:00] ---\nYour car registration expires on 20 March.\n\nQuestion: when does my car registration expire?",
     "rag_factual", "rag", _contains_facts(["20 March"])),
    # -- RAG refusal (2): the asked-about fact is NOT in context --
    ("--- NOTE 1 [Recorded: Tuesday, 03 Feb 2026 at 10:00] ---\nYour car registration expires on 20 March.\n\nQuestion: when does my library book need to be returned?",
     "rag_refusal", "rag", _is_rag_refusal),
    ("--- NOTE 1 [Recorded: Wednesday, 04 Mar 2026 at 11:00] ---\nYour gym membership renews on the 5th of each month.\n\nQuestion: how much is my water bill?",
     "rag_refusal", "rag", _is_rag_refusal),
]

assert len(VALIDATION_CASES) == 20

results = []
for user_content, category, mode, check in VALIDATION_CASES:
    answer = generate_rag(user_content) if mode == "rag" else generate(user_content)
    passed = check(answer)
    results.append((category, passed))
    mark = "PASS" if passed else "FAIL"
    preview = user_content if len(user_content) < 90 else user_content.split("Question:")[-1].strip()
    print(f"[{mark}] ({category}) {preview}\n       -> {answer}\n")

print("=" * 60)
by_category = {}
for category, passed in results:
    total, ok = by_category.get(category, (0, 0))
    by_category[category] = (total + 1, ok + (1 if passed else 0))
for category, (total, ok) in sorted(by_category.items()):
    print(f"{category:14s} {ok}/{total}")
overall = sum(1 for _, p in results if p)
print(f"{'TOTAL':14s} {overall}/{len(results)}")
if overall < len(results):
    print("\nNot a clean sweep -- worth reading the failures above before spending the export step.")
else:
    print("\nClean sweep. Proceed to export -- the real verdict is still the 103-case harness afterward.")

## 7. Export to GGUF (q4_k_m)

Matches the quantization the app already ships. This exports the merged LoRA weights the same way regardless of which trainer produced them. This step builds llama.cpp from source the first time and is the slowest cell in the notebook — 10-15 minutes is normal.

In [ ]:
model.save_pretrained_gguf(
    "qwen-task-extractor",
    tokenizer,
    quantization_method="q4_k_m",
)

!ls -lh qwen-task-extractor/*.gguf

In [ ]:
import glob
from google.colab import files

gguf = glob.glob("qwen-task-extractor/*.gguf")[0]
print("Downloading", gguf)
files.download(gguf)

## 8. Next steps, on your machine

```bash
# 1. Put the exported model where the harness looks
mv ~/Downloads/*.gguf models/qwen-task-extractor-q4_k_m.gguf

# 2. Score it against the frozen 103-case corpus, with the minimal prompt
#    -- this run also exercises preFilterZeroTaskNotes for third-party/
#    refusal cases, since runEval.ts's runCase() calls it exactly like
#    the app does.
XAYRA_EXTRACTION_PROMPT=minimal \
  npm run eval -- --model models/qwen-task-extractor-q4_k_m.gguf \
                  --corpus scripts/eval/corpus-full.jsonl
```

The bar to beat is **83.5% (86/103)** — the prompt-only stock baseline, still undefeated after five real training attempts (v2, v3, v5, v6, v7). Watch specifically:

- `third-party` and `refusal` — now decided by `preFilterZeroTaskNotes` in code, not this model. These categories should score however that pre-filter's own coverage allows (25/26 known cases, verified separately in `__tests__/tier1-extractionLogic.test.ts`), regardless of what this checkpoint does.
- `date-resolution`, `recurrence`, `attribution` (contrastive), `stt-noise` — every prior run held these near-perfect; this is what the fine-tune is actually responsible for now, with less budget spent trying (and failing) to also fix the categories above.
- `rag-factual`, `rag-refusal`, `rag-grounding` — v7 held these reasonably (4/4, 2/3, 2/4); check they haven't regressed with the narrower dataset.

If this checkpoint clears 83.5% with third-party/refusal no longer dragging it down, that's the first shippable result across the whole arc. If it doesn't, the fine-tuning side is no longer the suspect — the pre-filter's own coverage or the RAG/date/recurrence training itself would be.